In [2]:
import pandas as pd
import numpy as np
import random

# Semilla para reproducibilidad
np.random.seed(42)

# Parámetros para la generación de datos
dias = 720
regiones = {
    "Norte": {"promedio_ventas": 600, "promedio_promociones": 150, "crecimiento_diario": 0.001},
    "Sur": {"promedio_ventas": 450, "promedio_promociones": 100, "crecimiento_diario": 0.0008},
    "Este": {"promedio_ventas": 500, "promedio_promociones": 120, "crecimiento_diario": 0.0012},
    "Oeste": {"promedio_ventas": 550, "promedio_promociones": 130, "crecimiento_diario": 0.001},
    "Centro": {"promedio_ventas": 700, "promedio_promociones": 180, "crecimiento_diario": 0.0015},
}
categorias = {
    "Ropa Casual": 1.0,
    "Ropa Formal": 1.2,
    "Deportivo": 0.8,
    "Accesorios": 0.6,
}

# Fechas
fechas = pd.date_range(start="2021-01-01", periods=dias, freq="D")

# Función para generar descuentos realistas
def generar_descuento(fecha):
    if fecha.weekday() in [5, 6]:  # Fines de semana
        return np.random.choice([0, 10, 15, 20], p=[0.4, 0.3, 0.2, 0.1])
    elif fecha.day in range(25, 32):  # Fin de mes
        return np.random.choice([5, 10, 20, 30], p=[0.3, 0.3, 0.3, 0.1])
    else:
        return np.random.choice([0, 5, 10], p=[0.6, 0.3, 0.1])  # Días regulares

# Generar datos
data = []
for i, fecha in enumerate(fechas):
    for region, valores_region in regiones.items():
        for categoria, factor_categoria in categorias.items():
            # Ajustar promedios con crecimiento diario
            promedio_ventas = valores_region["promedio_ventas"] * (1 + valores_region["crecimiento_diario"])**i
            promedio_promociones = valores_region["promedio_promociones"] * (1 + valores_region["crecimiento_diario"])**i
            
            # Generar ventas y promociones ajustadas por región y categoría
            ventas_base = np.random.normal(promedio_ventas, 100)
            promociones_base = np.random.normal(promedio_promociones, 30)
            
            ventas = max(0, round(ventas_base * factor_categoria, 2))
            promociones = max(0, round(promociones_base * factor_categoria, 2))
            
            # Generar descuento para la fecha
            descuento = generar_descuento(fecha)
            
            # Relación entre stock disponible y ventas
            stock_disponible = max(50, int(np.random.normal(ventas * 0.5, 20)))  # Stock relacionado con ventas
            
            # Relación entre clientes atendidos y ventas
            clientes_atendidos = max(20, int(np.random.normal(ventas / 10, 5)))
            
            # Relación entre descuento y ventas
            if descuento > 0:
                ventas *= (1 + descuento / 100)
            
            # Agregar datos
            data.append([fecha, region, categoria, ventas, promociones, descuento, stock_disponible, clientes_atendidos])

# Crear DataFrame
df = pd.DataFrame(data, columns=["Fecha", "Región", "Categoría de Producto", "Ventas (USD)", "Gasto en Promociones (USD)", "Descuento (%)", "Stock Disponible (Unidades)", "Clientes Atendidos"])

# Introducir ruido adicional en los datos
df.loc[df.sample(frac=0.1).index, "Ventas (USD)"] *= np.random.uniform(0.9, 1.1)
df.loc[df.sample(frac=0.1).index, "Gasto en Promociones (USD)"] *= np.random.uniform(0.9, 1.1)

# Guardar en CSV
df.to_csv("data/modaexpress_ventas.csv", index=False)

print("Dataset generado y guardado como 'modaexpress_ventas.csv'")

Dataset generado y guardado como 'modaexpress_ventas.csv'
